# Glacier outputs postprocessing

In [ ]:
# spatial
import geopandas as gpd
import pandas as pd
import xarray as xr 
from pathlib import Path
from tqdm import tqdm

from src import glaciers
from src.utils import load_config
cfg = load_config()

In [ ]:
# lookup tables
andeangc_lookup   = gpd.read_file(cfg["dir"]["andeangc"] + "dataset/AndeanGC_shape.gpkg").set_index('gauge_id')[["geometry"]]
hydroatlas_lookup = gpd.read_file("data/hydro/basins_hydroATLAS_lev08.gpkg").set_index('gauge_id')[["geometry"]]

glaciers_lookup = pd.concat([gpd.read_file(cfg["dir"]["data"] + "/glacier_outlines/RGI60_16.shp").set_index('RGIId')[["geometry"]], 
                             gpd.read_file(cfg["dir"]["data"] + "/glacier_outlines/RGI60_17.shp").set_index('RGIId')[["geometry"]]])

## Glacier-wide aggregation

In [ ]:
# period 2000-2024
columns = ["rgi_subregion", "cenlat", "cenlon", "rgi_area_km2", "rgi_year", "inv_volume_km3"]
glacier_metadata = pd.read_csv(cfg["dir"]["pipeline"] + "/oggm/glacier_statistics.csv", low_memory=False, index_col=0)
glacier_metadata = glacier_metadata[columns]

df_historical = xr.open_dataset(cfg["dir"]["pipeline"] + "/oggm/run_output_historical.nc").volume
df_historical = df_historical.drop_vars(["hydro_month", "hydro_year", "calendar_year", "calendar_month"])
df_historical = df_historical.sel(time = 2020).to_pandas().rename("volume_2020_historical")

df_future = []
files = list(Path(cfg["dir"]["pipeline"]).glob("oggm/*output*"))
files = [f for f in files if "historical" not in f.name]
warming_levels = [f.name.split('_')[2] for f in files]

# period 2000-2099
for gcm in files:
    ts_future = xr.open_dataset(gcm).volume
    ts_future = ts_future.drop_vars(["hydro_month", "hydro_year", "calendar_year", "calendar_month"])
    ts_future = ts_future.sel(time = 2100).to_pandas().rename("volume_2100_{}".format(gcm.name.split('_')[2]))
    df_future.append(ts_future)

df_future = pd.concat(df_future, axis=1)
df_future = df_future.T.groupby(level=0).mean().T
df = pd.concat([glacier_metadata, df_historical, df_future], axis = 1)
df = df.dropna() # remove glaciers that failed to run in OGGM
df.to_csv("data/glacier/glacier_metadata_evolution.csv")


# list with glaciers that successfully ran in OGGM
glacier_sucess = df.index

## Region-wide aggregation

In [ ]:
# period 2000-2024
df_historical = xr.open_dataset(cfg["dir"]["pipeline"] + "/oggm/run_output_historical.nc")[["melt_on_glacier", "volume"]]
df_historical = df_historical.drop_vars(["hydro_month", "calendar_month"]).drop_vars(["hydro_year", "calendar_year"])
df_historical = df_historical.sel(rgi_id=glacier_sucess).sum(dim="rgi_id")
df_historical.to_netcdf("data/glacier/glacier_evolution_2000_2024.nc")

df_future = []
files = list(Path(cfg["dir"]["pipeline"]).glob("oggm/*output*"))
files = [f for f in files if "historical" not in f.name]
warming_levels = [f.name.split('_')[2] for f in files]

# period 2000-2099
for gcm in files:
    ts_future = xr.open_dataset(gcm)[["melt_on_glacier", "volume"]].load()
    ts_future = ts_future.drop_vars(["hydro_month", "calendar_month", "hydro_year", "calendar_year"])
    ts_future = ts_future.sel(rgi_id=glacier_sucess).sum(dim="rgi_id")
    df_future.append(ts_future)

df_future = xr.concat(df_future, dim = "warming_level")
df_future = df_future.assign_coords(warming_level=("warming_level", warming_levels))
df_future.to_netcdf("data/glacier/glacier_evolution_2025_2099.nc")

## Basin-wide aggregation

In [ ]:
hydroatlas_lookup = glaciers.calculate_basin_overlap(glaciers_lookup, hydroatlas_lookup)
andeangc_lookup   = glaciers.calculate_basin_overlap(glaciers_lookup, andeangc_lookup)

In [ ]:
paths = list(Path(cfg["dir"]["pipeline"]).glob("oggm/*output_historical*nc"))

for path in tqdm(paths):  

    # Load and preprocess data once
    oggm_ts = xr.open_dataset(path)[glaciers.variables].load()
    oggm_ts = oggm_ts.sel(time = slice(2000, 2024))
    oggm_ts = oggm_ts.sel(rgi_id=glacier_sucess)
    oggm_ts = glaciers.change_calendar(oggm_ts)
    oggm_ts = glaciers.hydro_variables(oggm_ts, only_melt=True)

    # for HydroATLAS and AndeanGC
    hydroatlas_lookup = hydroatlas_lookup[hydroatlas_lookup.index.isin(oggm_ts.rgi_id.values)]
    oggm_ts_hydroatlas = glaciers.aggregate_data_by_basin(oggm_ts, hydroatlas_lookup)
    andeangc_lookup = andeangc_lookup[andeangc_lookup.index.isin(oggm_ts.rgi_id.values)]
    oggm_ts_andeangc = glaciers.aggregate_data_by_basin(oggm_ts, andeangc_lookup)
    
    # Merge and save
    oggm_ts = xr.merge([oggm_ts_andeangc, oggm_ts_hydroatlas], join='outer', compat='no_conflicts').astype('float32')
    oggm_ts.to_netcdf("data/glacier/" + path.name, encoding={var: {"compression": "zstd"} for var in oggm_ts.data_vars})